In [ ]:
import json
from pathlib import Path
import re
import time

from tqdm.auto import tqdm

from schema_development.discovery import (
    load_prompt,
    analyze_snapshot,
    process_response,
)
from schema_development.io import load_json

# Inputs

In [ ]:
# Inputs
SYSTEM_PROMPT_PATH = Path("prompts/discovery_system.md")
USER_PROMPT_PATH = Path("prompts/discovery_user.md")
MODEL_NAME = "gpt-5.5"
SLEEP_SECONDS = 0.2  # safety throttle
SOURCE = "prwp"  # "unhcr" or "prwp" or "refugee"
N_SAMPLES = 30  # None if processing all
OUTPUT_PATH = Path("../../runs/discovery") / (
    f"schema_discovery_results_{SOURCE.lower()}.jsonl"
)

# Materialized by scripts/fetch_data.py.
SNAPSHOTS_DIR = Path("../../data/source/development/snapshots") / SOURCE
METADATA_DIR = Path("../../data/source/development/metadata") / SOURCE


# Load data

In [ ]:
system_prompt = load_prompt(SYSTEM_PROMPT_PATH)
user_prompt_template = load_prompt(USER_PROMPT_PATH)

In [ ]:
metadata_files = list(Path(METADATA_DIR).glob("*.json"))
metadata_lookup = {x.stem: x for x in metadata_files}
len(metadata_files)

In [ ]:
snapshot_files = list(Path(SNAPSHOTS_DIR).glob("*.png"))
len(snapshot_files)

In [ ]:
# For improvement: move sampling after removing items from skip list
# Sampling
if N_SAMPLES:
    import random
    figures = [x for x in snapshot_files if x.stem.split("_")[-2] == "figure"]
    tables = [x for x in snapshot_files if x.stem.split("_")[-2] == "table"]

    figures = random.sample(figures, N_SAMPLES)
    tables = random.sample(tables, N_SAMPLES)
    
    snapshot_files = figures + tables

len(snapshot_files)

# Main pipeline

In [ ]:
# Build skip list
to_skip = set()
if Path(OUTPUT_PATH).exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            to_skip.add(json.loads(line)["snapshot_id"])

In [ ]:
# Initialize dir
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
    for s in tqdm(snapshot_files):
        snapshot_id = s.name

        # Skip if in skip list
        if snapshot_id in to_skip:
            print(f"Skipped: {s.name}")
            continue

        # Base row — every row has the same keys
        row = {
            "snapshot_id": snapshot_id,
            "source": SOURCE,
            "model": MODEL_NAME,
            "raw_output": None,
            "usage": None,
            "cost": None,
            "status": None,
            "incomplete_details": None,
            "parsed_output": None,
            "error": None,
        }

        try:
            # Infer metadata file from snapshot filename
            parts = re.split("(_figure|_table)", s.stem)
            fname = parts[0]
            label = parts[1].lstrip("_")
            metadata_path = metadata_lookup.get(fname)
            if metadata_path:
                metadata = load_json(metadata_path)
            else:
                metadata = None

            # Create user prompt
            user_prompt = user_prompt_template.replace(
                "{DOCUMENT_METADATA_JSON}",
                json.dumps(metadata, indent=2, ensure_ascii=False),
            )

            # Responses API
            response = analyze_snapshot(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                image_path=str(s),
                model=MODEL_NAME,
                max_output_tokens=5000,
            )

            # Parse and process response
            result = process_response(response, model=MODEL_NAME)

            # Log data
            row["raw_output"] = result["raw_output"]
            row["usage"] = result["usage"]
            row["cost"] = result["cost"]
            row["status"] = result["status"]
            row["incomplete_details"] = result["incomplete_details"]
            row["parsed_output"] = result["parsed_output"]
            row["error"] = result["error"]  # None on success, message on parse failure

        except Exception as e:
            row["error"] = str(e)

        # Single write point — always executes
        out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
        out_f.flush()

        time.sleep(SLEEP_SECONDS)